<a href="https://colab.research.google.com/github/somaiah-ui/RAG-WITHOUT-LANGCHAIN/blob/main/RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 93.0 MB/s eta 0:00:00


In [2]:
import os
import torch
import faiss
import numpy as np
import gradio as gr

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


print("Loading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)


model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype="auto",
    device_map="auto"
)


print("Models loaded successfully!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Models loaded successfully!


In [4]:
vector_index = None
documents = []

In [5]:
def extract_pdf(path):

    reader = PdfReader(path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = page.extract_text() or ""

        text = " ".join(
            text.split()
        )

        if text.strip():

            pages.append(
                {
                    "text": text,
                    "source": Path(path).name,
                    "page": page_number
                }
            )

    return pages

In [6]:
def chunk_text(
    text,
    chunk_size=700,
    overlap=120
):

    if overlap >= chunk_size:

        overlap = chunk_size // 5


    chunks = []

    start = 0


    while start < len(text):

        end = min(
            start + chunk_size,
            len(text)
        )


        chunk = text[start:end]


        # Try to stop near a sentence
        if end < len(text):

            boundary = max(
                chunk.rfind(". "),
                chunk.rfind("? "),
                chunk.rfind("! "),
                chunk.rfind(" ")
            )


            if boundary > chunk_size * 0.6:

                end = start + boundary + 1

                chunk = text[start:end]


        chunk = chunk.strip()


        if chunk:

            chunks.append(chunk)


        if end >= len(text):

            break


        start = max(
            end - overlap,
            start + 1
        )


    return chunks

In [7]:
def build_knowledge_base(
    files,
    chunk_size=700,
    overlap=120
):

    global vector_index
    global documents


    if not files:

        return "Please upload at least one PDF."


    documents = []


    # Read every uploaded PDF
    for file in files:

        pages = extract_pdf(file)


        for page in pages:

            chunks = chunk_text(
                page["text"],
                int(chunk_size),
                int(overlap)
            )


            for chunk_number, chunk in enumerate(
                chunks,
                start=1
            ):

                documents.append(
                    {
                        "text": chunk,

                        "metadata": {

                            "source": page["source"],

                            "page": page["page"],

                            "chunk": chunk_number

                        }
                    }
                )


    if len(documents) == 0:

        return "No readable text found."


    print(
        "Total chunks:",
        len(documents)
    )


    # Get text from chunks
    texts = [

        document["text"]

        for document in documents

    ]


    print("Creating embeddings...")


    # Convert text into vectors
    embeddings = embedding_model.encode(

        texts,

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=True

    )


    embeddings = embeddings.astype(
        "float32"
    )


    # Get vector dimensions
    dimension = embeddings.shape[1]


    # Create FAISS database
    vector_index = faiss.IndexFlatIP(
        dimension
    )


    # Add embeddings
    vector_index.add(
        embeddings
    )


    print("FAISS index created!")


    return (
        f"Knowledge base ready!\n\n"
        f"Created {len(documents)} chunks "
        f"from {len(files)} PDF file(s)."
    )

In [8]:
def retrieve(
    query,
    top_k=4
):

    global vector_index
    global documents


    if vector_index is None:

        return []


    # Convert question into embedding
    query_embedding = embedding_model.encode(

        [query],

        convert_to_numpy=True,

        normalize_embeddings=True

    )


    query_embedding = query_embedding.astype(
        "float32"
    )


    # Search FAISS
    scores, indices = vector_index.search(

        query_embedding,

        min(
            int(top_k),
            len(documents)
        )
    )


    results = []


    for score, index in zip(
        scores[0],
        indices[0]
    ):


        if index == -1:

            continue


        document = documents[index]


        results.append(
            {

                "text":
                    document["text"],

                "metadata":
                    document["metadata"],

                "score":
                    float(score)

            }
        )


    return results

In [9]:
def build_prompt(
    question,
    retrieved_chunks
):

    context = []


    for i, item in enumerate(
        retrieved_chunks,
        start=1
    ):

        metadata = item["metadata"]


        context.append(
            f"""
[Source {i}]

File:
{metadata['source']}

Page:
{metadata['page']}

Content:
{item['text']}
"""
        )


    context_text = "\n\n".join(
        context
    )


    prompt = f"""
You are a helpful document question-answering assistant.

Answer the user's question using ONLY the context provided below.

Rules:

1. Do not invent information.

2. If the answer does not exist in the context, say:

"I couldn't find that information in the uploaded documents."

3. Explain the answer clearly.

4. Mention the source number when useful.

5. Keep the answer focused on the question.


CONTEXT:

{context_text}


QUESTION:

{question}
"""


    return prompt

In [10]:
def generate_answer(
    question,
    top_k=4
):

    global vector_index


    if vector_index is None:

        return (
            "Please upload PDFs and build "
            "the knowledge base first.",
            ""
        )


    # RETRIEVAL
    retrieved_chunks = retrieve(
        question,
        top_k
    )


    # AUGMENTATION
    prompt = build_prompt(
        question,
        retrieved_chunks
    )


    messages = [

        {
            "role": "system",

            "content":
            "You are a document question-answering assistant."
        },

        {
            "role": "user",

            "content": prompt
        }

    ]


    formatted_prompt = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True

    )


    inputs = tokenizer(

        formatted_prompt,

        return_tensors="pt"

    ).to(model.device)


    # GENERATION
    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=350,

            temperature=0.2,

            do_sample=True

        )


    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]


    answer = tokenizer.decode(

        generated_tokens,

        skip_special_tokens=True

    ).strip()


    # Show retrieved sources
    source_text = []


    for i, item in enumerate(
        retrieved_chunks,
        start=1
    ):

        metadata = item["metadata"]


        source_text.append(

            f"{i}. "
            f"{metadata['source']} | "
            f"Page {metadata['page']} | "
            f"Similarity: "
            f"{item['score']:.3f}"

        )


    source_text = "\n".join(
        source_text
    )


    return answer, source_text

In [11]:
with gr.Blocks(
    title="RAG Without LangChain"
) as demo:


    gr.Markdown(
        """
# 📚 RAG Without LangChain

Upload PDF documents and ask questions about them.

### Technologies

- PyPDF
- Sentence Transformers
- FAISS
- Hugging Face Transformers
- Qwen
- Gradio

**LangChain is NOT used.**
"""
    )


    with gr.Row():


        # LEFT SIDE
        with gr.Column():

            gr.Markdown(
                "## Upload Documents"
            )


            files = gr.File(

                label="Upload PDFs",

                file_types=[
                    ".pdf"
                ],

                file_count="multiple",

                type="filepath"

            )


            chunk_size = gr.Slider(

                minimum=300,

                maximum=1500,

                value=700,

                step=100,

                label="Chunk Size"

            )


            overlap = gr.Slider(

                minimum=0,

                maximum=300,

                value=120,

                step=20,

                label="Chunk Overlap"

            )


            build_button = gr.Button(
                "Build Knowledge Base"
            )


            status = gr.Textbox(

                label="Knowledge Base Status",

                interactive=False

            )


        # RIGHT SIDE
        with gr.Column():

            gr.Markdown(
                "## Ask Your PDF"
            )


            question = gr.Textbox(

                label="Question",

                placeholder=
                "Example: What is this document about?"

            )


            top_k = gr.Slider(

                minimum=1,

                maximum=8,

                value=4,

                step=1,

                label="Number of Retrieved Chunks"

            )


            ask_button = gr.Button(
                "Ask Question"
            )


            answer = gr.Markdown()


            sources = gr.Textbox(

                label="Retrieved Sources",

                lines=6,

                interactive=False

            )


    build_button.click(

        fn=build_knowledge_base,

        inputs=[
            files,
            chunk_size,
            overlap
        ],

        outputs=status

    )


    ask_button.click(

        fn=generate_answer,

        inputs=[
            question,
            top_k
        ],

        outputs=[
            answer,
            sources
        ]

    )

In [ ]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4594c41970b6d701a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Total chunks: 125
Creating embeddings...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

FAISS index created!
